# Moondream Binary on Reduced Dataset

## Setup and Imports

In [1]:
import os
os.environ['HF_HOME'] = '../cache'

In [2]:
import json
from PIL import Image

from nazi_symbols_classification.training.data_preparation import get_image_paths
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

## Data Preparation

In [3]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection-simple", ("train", "test", "val"))

In [4]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/val')]

In [5]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

## Model Loading and Classification

In [7]:
model = AutoModelForCausalLM.from_pretrained(
"vikhyatk/moondream2",
revision="2025-01-09",
trust_remote_code=True, # Uncomment for GPU acceleration & pip install accelerate # device_map={"": "cuda"}
device_map={"": "cuda"}
)

2025-06-08 01:48:42.321313: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [10]:
def classify_document(doc_path, prompts):
    image = Image.open(doc_path)
    encoded_image = model.encode_image(image)

    for prompt in prompts:
        answer = model.query(encoded_image, prompt)["answer"]
        if "Yes" in answer:
            return prompts[prompt]

    return "non-nazi"

In [8]:
prompts = {
    "Analyse whether the image contains a single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "nazi",
    "Analyse whether the image contains a skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "nazi",
    "Analyse whether the image contains a black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "nazi",
    "Analyse whether the image contains no nazi related content": "non-nazi",
}

In [11]:
classify_result = classify_document(test_images[0], prompts)
classify_result

'non-nazi'

In [12]:
%%time

result = []

for image_path in test_images:
    result.append(classify_document(image_path, prompts))

CPU times: user 4h 20min 26s, sys: 33.8 s, total: 4h 21min
Wall time: 3h 4min 4s


Store the results in a JSON file for later analysis.

In [13]:
to_store = dict(y_true=y_test, y_pred=result)

with open("moondream-output/moondream_result_binary_reduced.json", "w") as f:
    json.dump(to_store, f)

In [14]:
with open("moondream-output/moondream_result_binary_reduced.json", "r") as f:
    to_store = json.load(f)

Print the classification report and calculate metrics.

In [15]:
y_test, result = to_store["y_true"], to_store["y_pred"]

In [16]:
result = ["nazi" if label == "naz" else label for label in result]
y_test_binary = [label.removesuffix("-symbol") for label in y_test]
print(classification_report(y_test_binary, result, digits=3))

              precision    recall  f1-score   support

        nazi      0.840     0.851     0.846       148
    non-nazi      0.999     0.998     0.998     14813

    accuracy                          0.997     14961
   macro avg      0.919     0.925     0.922     14961
weighted avg      0.997     0.997     0.997     14961



In [17]:
y_test_binary_tmp = [label == "nazi" for label in y_test_binary]
result_tmp = [label == "nazi" for label in result]
roc_auc_score(y_test_binary_tmp, result_tmp), accuracy_score(y_test_binary_tmp, result_tmp)

(0.9248655764385192, 0.9969253392152931)